In [ ]:
# !pip install scikit-learn-extra
# !pip install "numpy<2.0"
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [ ]:
rng = np.random.default_rng(None)
alpha = rng.exponential(2.0)
n = rng.integers(10, 200)
# sample_crp_with_min_clusters(alpha, n)

In [ ]:
import numpy as np
from statsmodels.tsa.arima_process import ArmaProcess

def durbin_levinson(pacf):
    p = len(pacf)
    phi = np.zeros(p)
    phis = np.zeros((p,p))
    phi[0] = pacf[0]
    phis[0,0] = pacf[0]
    for k in range(1, p):
        sum_val = pacf[k]
        for j in range(k):
            sum_val -= phis[k-1, j] * pacf[k-j-1]
        phi[k] = sum_val
        for j in range(k):
            phis[k,j] = phis[k-1,j] - pacf[k] * phis[k-1,k-j-1]
        phis[k,k] = pacf[k]
    return phis[-1]

def sample_crp_with_min_clusters(alpha, n):
    """Generate cluster assignments for n items from a CRP with concentration alpha,
    conditioned on having more than 1 cluster."""
    while True:
        assignments = []
        cluster_counts = []
        for i in range(n):
            if i == 0:
                assignments.append(0)
                cluster_counts.append(1)
            else:
                probs = np.array(cluster_counts + [alpha])
                probs = probs / probs.sum()
                choice = np.random.choice(len(probs), p=probs)
                assignments.append(choice)
                if choice == len(cluster_counts):
                    cluster_counts.append(1)
                else:
                    cluster_counts[choice] += 1
        
        assignments = np.array(assignments)
        if len(np.unique(assignments)) > 1:
            return assignments
        # else repeat the simulation until more than one cluster is produced


def simulate_ar3_clustering_dataset_with_similarity_scenario2(series_length, random_state=None):
    rng = np.random.default_rng(random_state)
    
    # Sample number of time series n from discrete uniform [10, 200]
    n = rng.integers(10, 201)
    
    # Sample CRP concentration parameter alpha from Exp(1)
    alpha1 = rng.exponential(1.0)
    
    # Generate cluster assignments using CRP with concentration alpha1
    cluster_assignments = sample_crp_with_min_clusters(alpha1, n)
    
    # Identify distinct clusters and number of clusters K
    unique_clusters = np.unique(cluster_assignments)
    K = len(unique_clusters)
    
    # For each cluster, sample PACF and noise variance, convert to AR coefficients
    cluster_ar_coefs = []
    cluster_noise_vars = []
    for _ in range(K):
        pacf = rng.uniform(-1, 1, size=3)
        ar_coefs = durbin_levinson(pacf)
        ar_coefs_full = np.concatenate(([1], -ar_coefs))
        cluster_ar_coefs.append(ar_coefs_full)
        noise_var = rng.uniform(0.1, 2)
        cluster_noise_vars.append(noise_var)
    
    # Construct feature matrix by simulating time series and calculating autocorrelations
    features = []
    for i in range(n):
        # Map cluster label to index used in parameters list
        cluster_idx = np.where(unique_clusters == cluster_assignments[i])[0][0]
        arma_process = ArmaProcess(cluster_ar_coefs[cluster_idx], [1])
        ts = arma_process.generate_sample(nsample=series_length, scale=np.sqrt(cluster_noise_vars[cluster_idx]), burnin=100)
        acfs = [1.0]  # lag 0
        for lag in range(1, 4):
            if lag < len(ts):
                acf = np.corrcoef(ts[:-lag], ts[lag:])[0,1]
            else:
                acf = 0.0
            acfs.append(acf)
        features.append(acfs[1:])  # take lags 1,2,3

    D = np.array(features)                             # feature matrix (n, 3)
    C = cluster_assignments                            # cluster labels (n,)
    S = (C[:, None] == C[None, :]).astype(int)        # similarity matrix (n, n)
    
    return D, C, S

# Example usage
D, C, S = simulate_ar3_clustering_dataset_with_similarity_scenario2(series_length=100, random_state=42)
print("Number of series (n):", D.shape[0])
print("Number of features:", D.shape[1])
print("Clusters assigned:", len(np.unique(C)))
print("Similarity matrix shape:", S.shape)


In [ ]:
sample_crp_with_min_clusters(0.5, 20)

In [ ]:
import torch

def prepare_pairwise_data(D, S):
    ''' Returns pairs (concat) and targets for all i < j '''
    n = D.shape[0]
    pairs = []
    targets = []
    for i in range(n):
        for j in range(i+1, n):
            pair = np.concatenate([D[i], D[j]])  # shape (6,)
            target = S[i, j]                     # 1 if same cluster, 0 if not
            pairs.append(pair)
            targets.append(target)
    return np.array(pairs), np.array(targets)

# Example preparation:
pairs, targets = prepare_pairwise_data(D, S)
print("Pairs shape:", pairs.shape)       # (n*(n-1)/2, 6)
print("Targets shape:", targets.shape)   # (n*(n-1)/2,)
print(pairs)
print(targets)


In [ ]:
# Constructing the network

import torch
import torch.nn as nn

class PairwiseNN(nn.Module):
    def __init__(self, vector_dim=3, embed_dim=128, hidden_dim=256):
        super().__init__()
        # Embedding network phi applied independently to each vector
        self.phi = nn.Sequential(
            nn.Linear(vector_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU()
        )
        # Prediction network rho that takes aggregated embedding
        self.rho = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x shape: (batch_size, 6) where each pair concat of two 3-d vectors
        v1 = x[:, :3]  # first vector
        v2 = x[:, 3:]  # second vector

        emb1 = self.phi(v1)  # embed first vector
        emb2 = self.phi(v2)  # embed second vector

        combined = emb1 + emb2  # symmetric aggregation (sum)

        out = self.rho(combined)  # predict
        return out.squeeze(-1)  # shape: (batch_size,)

# Example initialization:
model = PairwiseNN(vector_dim=3, embed_dim=128, hidden_dim=256)



In [ ]:
# Training

import torch
import torch.nn as nn
import numpy as np

# Assuming PairwiseNN and prepare_pairwise_data are already defined,
# and simulate_ar3_clustering_dataset_with_similarity_scenario2 is implemented

series_lengths = [300]
num_epochs = 1
units_per_epoch = 2000000  # total simulated datasets per epoch
datasets_per_batch = 64   # datasets per batch update
early_stopping_patience = 100
models_by_length = {}

for T in series_lengths:
    print(f"\nTraining model for series length: {T}")
    model = PairwiseNN(vector_dim=3, embed_dim=128, hidden_dim=256)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()
    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(num_epochs):
        losses = []
        indices = np.arange(units_per_epoch // datasets_per_batch)
        for _ in indices:

            batch_num = len(losses) + 1
            if batch_num % 100 == 0:
               print("Batch", batch_num)
            
            batch_pairs = []
            batch_targets = []
            for _ in range(datasets_per_batch):
                # Scenario 2 simulation: variable n and K internally handled
                D, C, S = simulate_ar3_clustering_dataset_with_similarity_scenario2(series_length=T)
                pairs, targets = prepare_pairwise_data(D, S)
                batch_pairs.append(pairs)
                batch_targets.append(targets)
            
            pairs_tensor = torch.tensor(np.vstack(batch_pairs)).float()
            targets_tensor = torch.tensor(np.hstack(batch_targets)).float()
            model.train()
            optimizer.zero_grad()
            outputs = model(pairs_tensor)
            loss = criterion(outputs, targets_tensor)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        mean_loss = np.mean(losses)

        # Validation step
        model.eval()
        Dv, Cv, Sv = simulate_ar3_clustering_dataset_with_similarity_scenario2(series_length=T)
        pairs_val, targets_val = prepare_pairwise_data(Dv, Sv)
        pairs_tensor_val = torch.tensor(pairs_val).float()
        targets_tensor_val = torch.tensor(targets_val).float()
        with torch.no_grad():
            outputs_val = model(pairs_tensor_val)
            val_loss = criterion(outputs_val, targets_tensor_val).item()

        print(f"Epoch {epoch+1}, train loss: {mean_loss:.4f}, val loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            # You can optionally save model state here:
            # best_model_state = model.state_dict()
        else:
            patience_counter += 1

        if patience_counter >= early_stopping_patience:
            print(f"Early stopping at epoch {epoch+1} (best val loss {best_val_loss:.4f})")
            break

    models_by_length[f"model_{T}"] = model

print("\nTraining completed. Models saved in memory.")

In [ ]:
# Evaluation

import torch
import numpy as np
import networkx as nx
import community.community_louvain as community_louvain
import pandas as pd
from sklearn.cluster import SpectralClustering, KMeans, AgglomerativeClustering
from sklearn_extra.cluster import KMedoids
from sklearn.metrics import adjusted_rand_score

num_trials = 10000
series_lengths = [300]

results = {T: {
    "spectral": [],
    "kmeans": [],
    "kmedoids": [],
    "agglo": [],
    "louvain_pred": []
} for T in series_lengths}

for T in series_lengths:
    model = models_by_length[f"model_{T}"]
    print(f"\nEvaluating model trained on series length {T}")
    for trial in range(num_trials):
        # Scenario 2 simulation: returns variable n and K internally
        D_test, C_test, S_test = simulate_ar3_clustering_dataset_with_similarity_scenario2(series_length=T)
        
        n_series = D_test.shape[0]
        K_true = len(np.unique(C_test))  # true number of clusters per dataset

        pairs, targets = prepare_pairwise_data(D_test, S_test)
        pairs_tensor = torch.tensor(pairs).float()
        
        with torch.no_grad():
            pred_sim_probs = model(pairs_tensor).detach().cpu().numpy()
            pred_sim_matrix = np.zeros((n_series, n_series), dtype=float)
            idx = 0
            for i in range(n_series):
                for j in range(i + 1, n_series):
                    pred_sim_matrix[i, j] = pred_sim_probs[idx]
                    pred_sim_matrix[j, i] = pred_sim_probs[idx]
                    idx += 1
            np.fill_diagonal(pred_sim_matrix, 1.0)

            spectral = SpectralClustering(n_clusters=K_true, affinity="precomputed", random_state=trial)
            spectral_labels = spectral.fit_predict(pred_sim_matrix)
            spectral_ari = adjusted_rand_score(C_test, spectral_labels)
            results[T]["spectral"].append(spectral_ari)

            kmeans = KMeans(n_clusters=K_true, n_init=200, random_state=trial)
            kmeans_labels = kmeans.fit_predict(D_test)
            kmeans_ari = adjusted_rand_score(C_test, kmeans_labels)
            results[T]["kmeans"].append(kmeans_ari)

            kmedoids = KMedoids(n_clusters=K_true, random_state=trial, init="k-medoids++")
            kmedoids_labels = kmedoids.fit_predict(D_test)
            kmedoids_ari = adjusted_rand_score(C_test, kmedoids_labels)
            results[T]["kmedoids"].append(kmedoids_ari)

            agglo = AgglomerativeClustering(n_clusters=K_true, linkage="ward")
            agglo_labels = agglo.fit_predict(D_test)
            agglo_ari = adjusted_rand_score(C_test, agglo_labels)
            results[T]["agglo"].append(agglo_ari)

            G_pred = nx.from_numpy_array(pred_sim_matrix)
            partition_pred = community_louvain.best_partition(G_pred, random_state=trial)
            louvain_pred_labels = np.array([partition_pred[i] for i in range(n_series)])
            louvain_pred_ari = adjusted_rand_score(C_test, louvain_pred_labels)
            results[T]["louvain_pred"].append(louvain_pred_ari)

        if (trial + 1) % 100 == 0:
            print(f"Trial {trial + 1}/{num_trials} completed for series length {T}")

# Prepare summary table
summary_data = {
    "Series Length": [],
    "Spectral Clustering": [],
    "Louvain on Affinities": [],
    "KMeans on Features": [],
    "KMedoids on Features": [],
    "Agglomerative (Ward)": [],
}
for T in series_lengths:
    summary_data["Series Length"].append(T)
    summary_data["Spectral Clustering"].append(np.mean(results[T]["spectral"]))
    summary_data["Louvain on Affinities"].append(np.mean(results[T]["louvain_pred"]))
    summary_data["KMeans on Features"].append(np.mean(results[T]["kmeans"]))
    summary_data["KMedoids on Features"].append(np.mean(results[T]["kmedoids"]))
    summary_data["Agglomerative (Ward)"].append(np.mean(results[T]["agglo"]))

df_summary = pd.DataFrame(summary_data)
import os
# output_dir = "df_summary_scenario_2"
# os.makedirs(output_dir, exist_ok=True)
# df_summary.to_csv(f'{output_dir}/df_summary_scenario_2.csv', index=False)
print(df_summary.to_string(index=False, float_format='%.3f'))


In [ ]:
import matplotlib.pyplot as plt

csv_path = 'df_summary_scenario_2/df_summary_scenario_2.csv'
import pandas as pd

df_summary = pd.read_csv(csv_path)

plt.figure(figsize=(8, 6))

plt.plot(df_summary["Series Length"], df_summary["Spectral Clustering"], marker='o', label='Proposed (spectral)', color='blue')
plt.plot(df_summary["Series Length"], df_summary["Louvain on Affinities"], marker='s', label='Proposed (Louvain)', color='orange')
plt.plot(df_summary["Series Length"], df_summary["KMeans on Features"], marker='^', label='K-Means', color='green')
plt.plot(df_summary["Series Length"], df_summary["KMedoids on Features"], marker='v', label='K-Medoids', color='red')
plt.plot(df_summary["Series Length"], df_summary["Agglomerative (Ward)"], marker='D', label='Hierarchical', color='purple')

plt.xlabel('Series length', fontsize=16)
plt.ylabel('ARI', fontsize=16)
plt.title('Scenario 2', fontsize=18)
plt.legend(fontsize=14)
plt.grid(True)
plt.xticks(df_summary["Series Length"], fontsize=14)
plt.yticks(fontsize=14)
plt.ylim(0.35, 1.0)

plt.savefig('fig1.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Plotting clustering accuracy with respect to the amount of training data

# Plotting clustering accuracy with respect to the amount of training data

import numpy as np
import matplotlib.pyplot as plt

training_data = np.array([20000, 50000, 100000, 200000, 500000, 1000000, 2000000])
spectral_50 = np.array([0.617, 0.703, 0.745, 0.794, 0.819, 0.831, 0.838])
spectral_150 = np.array([0.725, 0.781, 0.808, 0.842, 0.887, 0.899, 0.900])
spectral_300 = np.array([0.769, 0.800, 0.844, 0.876, 0.899, 0.904, 0.916])
louvain_50 = np.array([0.457, 0.501, 0.533, 0.557, 0.583, 0.590, 0.593])
louvain_150 = np.array([0.516, 0.566, 0.593, 0.614, 0.651, 0.655, 0.673])
louvain_300 = np.array([0.541, 0.580, 0.615, 0.649, 0.672, 0.674, 0.700])

# Create subplots side by side - massive figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(30, 12))

# Massive global font sizes
plt.rcParams.update({'font.size': 28})

# Power notation labels for training_data
xticklabels = ['2e4', '5e4', '1e5', '2e5', '5e5', '1e6', '2e6']

# Spectral Clustering Plot (left, all blue) - solid, dashed, dotted
ax1.plot(training_data, spectral_50, 'o-', color='blue', label='Series length = 50', linewidth=5, markersize=15)
ax1.plot(training_data, spectral_150, 's--', color='blue', label='Series length = 150', linewidth=5, markersize=15)
ax1.plot(training_data, spectral_300, '^:', color='blue', label='Series length = 300', linewidth=5, markersize=15)
ax1.set_xscale('log')
ax1.set_xticks(training_data)
ax1.set_xticklabels(xticklabels, rotation=50)
ax1.set_xlabel('Number of time series collections (N)', fontsize=45)
ax1.set_ylabel('ARI', fontsize=45)
ax1.set_title('Scenario 2 (spectral)', fontsize=55)
ax1.set_ylim(0.4, 1.0)
ax1.legend(fontsize=35)
ax1.tick_params(axis='both', labelsize=36)  # Increased tick label size from 28 to 36
ax1.grid(True, alpha=0.3)

# Louvain Clustering Plot (right, all orange) - solid, dashed, dotted
ax2.plot(training_data, louvain_50, 'o-', color='orange', label='Series length = 50', linewidth=5, markersize=15)
ax2.plot(training_data, louvain_150, 's--', color='orange', label='Series length = 150', linewidth=5, markersize=15)
ax2.plot(training_data, louvain_300, '^:', color='orange', label='Series length = 300', linewidth=5, markersize=15)
ax2.set_xscale('log')
ax2.set_xticks(training_data)
ax2.set_xticklabels(xticklabels, rotation=50)
ax2.set_xlabel('Number of time series collections (N)', fontsize=45)
ax2.set_ylabel('ARI', fontsize=45)
ax2.set_title('Scenario 2 (Louvain)', fontsize=55)
ax2.set_ylim(0.4, 1.0)
ax2.legend(fontsize=35)
ax2.tick_params(axis='both', labelsize=36)  # Increased tick label size from 28 to 36
ax2.grid(True, alpha=0.3)

plt.tight_layout()

# Save the figure
plt.savefig('fig2.png', dpi=300, bbox_inches='tight')

plt.show()